# Pokemon TCG AI Battle — Agent Testing

Run inside a Kaggle competition notebook (cabt is pre-installed there).

1. Set `GITHUB_URL`.
2. Edit `AGENTS`, `NICKNAMES`, and `N_GAMES` in section 4.
3. Run all cells.

## 1. Clone repo

In [ ]:
GITHUB_URL = "https://github.com/<YOUR_USERNAME>/<YOUR_REPO>.git"  # replace this
BRANCH     = None  # set to a branch name (e.g. "dev") to clone only that branch, or None for the default

!rm -rf /kaggle/working/repo
if BRANCH:
    !git clone --branch {BRANCH} --single-branch {GITHUB_URL} /kaggle/working/repo
else:
    !git clone {GITHUB_URL} /kaggle/working/repo

In [ ]:
import os, sys
os.chdir("/kaggle/working/repo")
# src/ holds both the ptcg/ and cg/ packages
sys.path.insert(0, "/kaggle/working/repo/src")

## 2. Card database

Loads typed card/attack data via `cg.api`.  
**If this fails, `RuleBasedAgent` is identical to `RandomAgent`** — every rule
that needs card data falls back to `RandomFallback`.

In [ ]:
from ptcg import card_db

loaded = card_db.load()
print(f"card_db loaded : {loaded}")

if loaded:
    print(f"Cards          : {len(card_db._cards)}")
    print(f"Attacks        : {len(card_db._attacks)}")
    sample = card_db.get_card(721)
    if sample:
        print(f"Sample (721)   : {sample.name}  "
              f"HP={sample.hp}  "
              f"type={sample.energyType}  "
              f"weakness={sample.weakness}")
else:
    print("WARNING: card_db failed to load. RuleBasedAgent will behave identically to RandomAgent.")

## 3. Deck contents

In [ ]:
from collections import Counter
from kaggle_environments.envs.cabt.cabt import deck as DEFAULT_DECK
from ptcg.rules.schema import CardType

TYPE_LABEL = {
    CardType.POKEMON:        "Pokemon",
    CardType.ITEM:           "Item",
    CardType.TOOL:           "Tool",
    CardType.SUPPORTER:      "Supporter",
    CardType.STADIUM:        "Stadium",
    CardType.BASIC_ENERGY:   "Basic Energy",
    CardType.SPECIAL_ENERGY: "Special Energy",
}

counts   = Counter(DEFAULT_DECK)
sections = {}

for card_id, count in sorted(counts.items()):
    card = card_db.get_card(card_id) if loaded else None
    if card:
        name, ctype = card.name, card.cardType
        hp = card.hp if card.cardType == CardType.POKEMON else None
    else:
        name, ctype, hp = f"Card#{card_id}", -1, None
    sections.setdefault(ctype, []).append((count, name, hp))

print(f"Deck — {len(DEFAULT_DECK)} cards\n")
for ctype, entries in sections.items():
    label = TYPE_LABEL.get(ctype, f"Type#{ctype}")
    total = sum(c for c, _, _ in entries)
    print(f"  {label} ({total})")
    print(f"  {'─' * 40}")
    for count, name, hp in sorted(entries, key=lambda x: x[1]):
        hp_str = f"  HP {hp}" if hp is not None else ""
        print(f"    {count}x  {name}{hp_str}")
    print()

## 4. Configure tournament

- `AGENTS` — class names to include in the tournament.
- `NICKNAMES` — display name shown in replays and logs for each agent.
- `N_GAMES` — games per matchup.

In [ ]:
AGENTS = [
    "RandomAgent",
    "RuleBasedAgent",
]

NICKNAMES = {
    "RandomAgent":    "Rookie",
    "RuleBasedAgent": "Rule Master",
}

N_GAMES = 20


def nickname(agent_class: str) -> str:
    """Return the display name for an agent class."""
    return NICKNAMES.get(agent_class, agent_class)

## 5. Run tournament

In [ ]:
import importlib, re
from collections import defaultdict
from kaggle_environments import make


def load_agent(class_name: str):
    module_name = re.sub(r"(?<!^)(?=[A-Z])", "_", class_name).lower()
    module = importlib.import_module(f"ptcg.agents.{module_name}")
    return getattr(module, class_name)()


def run_battle(agent0, agent1):
    env = make("cabt", configuration={"decks": [agent0.get_deck(), agent1.get_deck()]})
    env.run([agent0, agent1])
    rewards = [step["reward"] for step in env.steps[-1]]
    return rewards, env


records      = defaultdict(lambda: {"wins": 0, "draws": 0, "losses": 0})
rule_logs    = defaultdict(list)
matchup_envs = {}   # (class0, class1) -> env of the last game of that matchup

for name0 in AGENTS:
    for name1 in AGENTS:
        key  = (name0, name1)
        nick = f"{nickname(name0)} vs {nickname(name1)}"
        for _ in range(N_GAMES):
            a0, a1 = load_agent(name0), load_agent(name1)
            a0.on_game_start()
            a1.on_game_start()
            rewards, env = run_battle(a0, a1)
            matchup_envs[key] = env
            result = {"steps": env.steps, "rewards": rewards}
            a0.on_game_end(result)
            a1.on_game_end(result)
            if hasattr(a0, "rule_log"):  rule_logs[name0].extend(a0.rule_log)
            if hasattr(a1, "rule_log"):  rule_logs[name1].extend(a1.rule_log)
            r0 = rewards[0]
            if   r0 == 1:  records[key]["wins"]   += 1
            elif r0 == 0:  records[key]["draws"]  += 1
            else:          records[key]["losses"] += 1

        w, d, l = records[key]["wins"], records[key]["draws"], records[key]["losses"]
        print(f"{nick:<35}  {w}W {d}D {l}L  ({w/N_GAMES*100:.0f}% win rate for {nickname(name0)})")

## 6. Results summary

In [ ]:
print(f"{'Agent 0':<20} {'Agent 1':<20} {'W':>4} {'D':>4} {'L':>4} {'Win%':>6}")
print("-" * 65)
for (name0, name1), r in records.items():
    w, d, l = r["wins"], r["draws"], r["losses"]
    total   = w + d + l
    rate    = w / total * 100 if total > 0 else 0
    print(f"{nickname(name0):<20} {nickname(name1):<20} {w:>4} {d:>4} {l:>4} {rate:>5.0f}%")

## 7. Rule firing breakdown

If `RandomFallback` is near 100%, `card_db` failed to load — check section 2.

In [ ]:
from collections import Counter

for agent_name, log in rule_logs.items():
    if not log:
        continue
    total  = len(log)
    counts = Counter(log)
    print(f"\n{nickname(agent_name)}  ({total} decisions across all games)")
    print("─" * 55)
    for rule_name, count in counts.most_common():
        pct = count / total * 100
        bar = "█" * int(pct / 3)
        print(f"  {rule_name:<25} {count:>5}  ({pct:>5.1f}%)  {bar}")

## 8. Replay a tournament game

Set `REPLAY` to any matchup played in section 5 (any pair from `AGENTS`).

In [ ]:
REPLAY = ("RandomAgent", "RuleBasedAgent")

from ptcg.log_formatter import format_history

replay_env = matchup_envs.get(REPLAY)
if replay_env is None:
    print(f"No game recorded for {REPLAY}.")
    print(f"Available matchups: {list(matchup_envs.keys())}")
else:
    history = replay_env.steps[0][0].get("visualize")
    names   = [nickname(REPLAY[0]), nickname(REPLAY[1])]
    if history is None:
        print("No visualize data available.")
    else:
        print(format_history(history, agent_names=names))

## 9. Custom game

Run and replay a one-off game between any two agents. Does not affect the
tournament records above.

In [ ]:
# ── configure ─────────────────────────────────────────────────────────────────
CUSTOM_AGENT_0    = "RuleBasedAgent"
CUSTOM_AGENT_1    = "RandomAgent"
CUSTOM_NICKNAME_0 = "Challenger"
CUSTOM_NICKNAME_1 = "Opponent"
# ──────────────────────────────────────────────────────────────────────────────

from ptcg.log_formatter import format_history

ca0, ca1 = load_agent(CUSTOM_AGENT_0), load_agent(CUSTOM_AGENT_1)
ca0.on_game_start()
ca1.on_game_start()
custom_rewards, custom_env = run_battle(ca0, ca1)
ca0.on_game_end({"rewards": custom_rewards})
ca1.on_game_end({"rewards": custom_rewards})

r0, r1 = custom_rewards
winner = CUSTOM_NICKNAME_0 if r0 == 1 else (CUSTOM_NICKNAME_1 if r1 == 1 else "Draw")
print(f"{CUSTOM_NICKNAME_0} vs {CUSTOM_NICKNAME_1}  →  winner: {winner}\n")

history = custom_env.steps[0][0].get("visualize")
if history is None:
    print("No visualize data available.")
else:
    print(format_history(history, agent_names=[CUSTOM_NICKNAME_0, CUSTOM_NICKNAME_1]))